# 04 — STT and TTS: comparing free-tier models

**STT section:** sends the 16 kHz and 8 kHz clips to several STT options and compares
transcript, latency, and word error rate (needs `GROUND_TRUTH` set to what you actually said).

**TTS section:** synthesizes the same sentence with several TTS options and saves each as a
file in `audio_lab/outputs/tts/` so you can listen and judge quality yourself (accuracy isn't
measurable the way STT is).

Providers with no API key set are skipped automatically, so this runs even if you've only
filled in `GROQ_API_KEY`. To try more, add the key to `.env` (see `.env.example`):
- `HUGGINGFACE_API_KEY` — free tier at huggingface.co/settings/tokens
- `ELEVENLABS_API_KEY` — free tier at elevenlabs.io (limited characters/month)

**NVIDIA (Riva/NIM) ASR and TTS are deliberately left as stubs below, not real calls.**
NVIDIA's hosted audio models don't use the same simple REST shape as their LLM endpoints, and
I don't want to hand you a fabricated endpoint that silently fails. Check
https://build.nvidia.com for the current ASR/TTS model page and its "code" tab when you get
there, then fill in the stub.

In [ ]:
def tts_edge(text: str, out_path: str, voice: str = "en-US-AriaNeural") -> None:
    """Microsoft Edge voices. Free, no API key. Good quality, async under the hood.
    Jupyter already runs an event loop, so plain asyncio.run() fails there with
    'cannot be called from a running event loop' - nest_asyncio patches that."""
    import asyncio
    import edge_tts
    import nest_asyncio

    nest_asyncio.apply()

    async def _run():
        communicate = edge_tts.Communicate(text, voice)
        await communicate.save(out_path)

    asyncio.run(_run())

## Just TTS: Groq and ElevenLabs

A smaller, standalone block using only these two TTS providers, separate from the full
comparison harness above. Uses the same `GROQ_API_KEY` and `ELEVENLABS_API_KEY` from `.env`.

In [ ]:
import os

import requests
from groq import Groq

from backend.config import settings  # reads GROQ_API_KEY / ELEVENLABS_API_KEY from .env

# ---- constants ----
TTS_TEXT = "Hi, thanks for calling Saffron and Seoul. Would you like a table or a delivery order?"
OUTPUT_DIR = "../outputs/tts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Groq retired 'playai-tts'; current preview model is 'canopylabs/orpheus-v1-english'.
# voice="tara" is a guess from the open-source Orpheus model's usual voice tags - if this
# errors on the voice name, check console.groq.com/docs/text-to-speech for the confirmed list.
GROQ_MODEL = "canopylabs/orpheus-v1-english"
GROQ_VOICE = "tara"
ELEVENLABS_VOICE_ID = "21m00Tcm4TlvDq8ikWAM"  # "Rachel", a default ElevenLabs voice


# ---- Groq (Orpheus TTS) ----
def groq_tts(text: str, out_path: str, voice: str = GROQ_VOICE) -> None:
    client = Groq(api_key=settings.groq_api_key)
    response = client.audio.speech.create(
        model=GROQ_MODEL,
        voice=voice,
        input=text,
        response_format="wav",
    )
    response.write_to_file(out_path)


# ---- ElevenLabs ----
def elevenlabs_tts(text: str, out_path: str, voice_id: str = ELEVENLABS_VOICE_ID) -> None:
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"
    headers = {
        "xi-api-key": settings.elevenlabs_api_key,
        "Content-Type": "application/json",
    }
    payload = {"text": text, "model_id": "eleven_multilingual_v2"}
    resp = requests.post(url, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(resp.content)


# ---- run both ----
groq_path = os.path.join(OUTPUT_DIR, "groq_orpheus.wav")
groq_tts(TTS_TEXT, groq_path)
print(f"Groq Orpheus saved: {groq_path}")

elevenlabs_path = os.path.join(OUTPUT_DIR, "elevenlabs.mp3")
elevenlabs_tts(TTS_TEXT, elevenlabs_path)
print(f"ElevenLabs saved: {elevenlabs_path}")

print("\nListen to both files and compare quality yourself.")